<a href="https://colab.research.google.com/github/perezbrotonsluis/movie-tv-recommender/blob/main/notebooks/04_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Evaluación del Modelo Recomendador TF-IDF

Este notebook tiene como objetivo principal la evaluación del rendimiento de un modelo recomendador basado en TF-IDF. A través de dos enfoques complementarios, la evaluación cualitativa y la evaluación automática basada en géneros, buscaremos comprender la efectividad del modelo para sugerir contenido relevante.

Se cargarán los componentes del modelo (vectorizador TF-IDF y matriz de similitud) y se aplicarán métodos para obtener recomendaciones, permitiendo tanto una inspección humana de los resultados como una métrica cuantitativa de su precisión.

##Preparación

In [1]:
import pandas as pd

movie_tv_clean_url = "https://raw.githubusercontent.com/perezbrotonsluis/movie-tv-recommender/main/data/processed/movie_tv_clean.csv"
df_model = pd.read_csv(movie_tv_clean_url)


title_to_index = pd.Series(df_model.index, index=df_model['title'])

Este proceso de carga ha sido realizado con la ayuda de gemini.

In [2]:
import joblib
import requests
import io

github_url = "https://github.com/perezbrotonsluis/movie-tv-recommender/blob/main/models/vectorizador_tfidf.joblib" # @param {type: "string"}

# 1. Limpieza de la URL
raw_url = github_url.replace("github.com", "raw.githubusercontent.com").replace("/blob/", "/")

if "YOUR_GITHUB_URL_HERE" in raw_url or not raw_url:
    print("Please provide a valid GitHub URL for the .joblib file.")
else:
    try:
        # 2. Descarga con la URL corregida
        response = requests.get(raw_url)
        response.raise_for_status()

        # 3. Carga el modelo
        tfidf_vectorizer = joblib.load(io.BytesIO(response.content))
        print("TF-IDF vectorizer model loaded successfully!")

    except requests.exceptions.RequestException as e:
        print(f"Error downloading the model: {e}")
    except Exception as e:
        # Si el error es 'invalid load key', confirma que la URL es realmente el archivo binario
        print(f"An unexpected error occurred: {e}")

TF-IDF vectorizer model loaded successfully!


In [3]:
github_url = "https://github.com/perezbrotonsluis/movie-tv-recommender/blob/main/models/tfidf_matrix.joblib" # @param {type: "string"}

# 1. Limpieza de la URL
raw_url = github_url.replace("github.com", "raw.githubusercontent.com").replace("/blob/", "/")

if "YOUR_GITHUB_URL_HERE" in raw_url or not raw_url:
    print("Please provide a valid GitHub URL for the .joblib file.")
else:
    try:
        # 2. Descarga con la URL corregida
        response = requests.get(raw_url)
        response.raise_for_status()

        # 3. Carga el modelo
        tfidf_matrix = joblib.load(io.BytesIO(response.content))
        print("TF-IDF matrix loaded successfully!")

    except requests.exceptions.RequestException as e:
        print(f"Error downloading the model: {e}")
    except Exception as e:
        # Si el error es 'invalid load key', confirma que la URL es realmente el archivo binario
        print(f"An unexpected error occurred: {e}")

TF-IDF matrix loaded successfully!


In [4]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim_matrix = cosine_similarity(tfidf_matrix)

print("Tamaño de la matriz de similitud del coseno:", cosine_sim_matrix.shape)

Tamaño de la matriz de similitud del coseno: (19661, 19661)


##Evaluación cualitativa
Para esta evaluación cualitativa se han seleccionado 10 títulos muy conocidos por el público general (5 películas y 5 series). Con cada uno de estos el sistema generarán las 5 recomendaciones más relevantes y sobre ellas se realizará un análisis de los resultados, analizando las sugerencias del modelo.


In [5]:
def get_recommendations(title, cosine_sim_matrix, df_model, title_to_index, top_n=10):
    # Verifica si el título existe en nuestro índice
    if title not in title_to_index.index:
        return f"Title '{title}' not found in the dataset."

    # Obtiene el índice del título que coincide con la entrada.
    # Si title_to_index[title] devuelve una Serie (debido a títulos duplicados),
    # tomamos el primer índice por simplicidad.
    idx_potential_series = title_to_index[title]
    if isinstance(idx_potential_series, pd.Series):
        idx = idx_potential_series.iloc[0] # Toma el primer índice si existen varios
    else:
        idx = idx_potential_series

    # Obtiene los scores de similitud por pares para todos los títulos con ese título
    sim_scores = list(enumerate(cosine_sim_matrix[idx]))

    # Ordena los títulos basándose en los scores de similitud
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Obtiene los scores de los top_n títulos más similares (excluyendo el propio)
    sim_scores = sim_scores[1:top_n+1] # +1 porque el primero es el propio título

    # Obtiene los índices de los títulos
    movie_indices = [i[0] for i in sim_scores]

    # Obtiene los scores de similitud
    similarity_scores = [i[1] for i in sim_scores]

    # Devuelve los top_n títulos más similares junto con sus scores de similitud
    recommendations = df_model['title'].iloc[movie_indices]
    return pd.DataFrame({'Recommended Title': recommendations, 'Similarity Score': similarity_scores})

In [6]:
# Prueba con algunos títulos representativos
test_titles = [
    'Forrest Gump',
    'The Lord of the Rings: The Fellowship of the Ring',
    'Inception',
    'Blade Runner 2049',
    'Seven',
    'The Office',
    'The Mandalorian',
    'Chernobyl',
    'The Boys',
    'Stranger Things'
]

print("--- INICIANDO EVALUACIÓN CUALITATIVA ---")

for title in test_titles:
    print(f"\nRecomendaciones para '{title}':")
    recommendations = get_recommendations(title, cosine_sim_matrix, df_model, title_to_index, top_n=5)
    if isinstance(recommendations, str):
        print(recommendations)
    else:
        print(recommendations)

--- INICIANDO EVALUACIÓN CUALITATIVA ---

Recomendaciones para 'Forrest Gump':
         Recommended Title  Similarity Score
14245          Dark Waters          0.116172
16771      Marriage Palace          0.100092
5737   Alternate Realities          0.093469
2219             Cast Away          0.087907
2774        Staying Afloat          0.087321

Recomendaciones para 'The Lord of the Rings: The Fellowship of the Ring':
                                       Recommended Title  Similarity Score
12274  The Lord of the Rings: The Fellowship of the R...          0.932172
12183              The Lord of the Rings: The Two Towers          0.461239
12234      The Lord of the Rings: The Return of the King          0.459572
12716                  The Hobbit: An Unexpected Journey          0.256048
12919          The Hobbit: The Battle of the Five Armies          0.201418

Recomendaciones para 'Inception':
           Recommended Title  Similarity Score
14751  The Dark Knight Rises          0.1405

###Análisis de las películas
1. Forrest Gump

En este caso, la única recomendación con valor real es Cast Away. Aunque las tramas difieren, el modelo logra establecer una conexión gracias a que el actor Tom Hanks es el protagonista en ambas películas. El resto de sugerencias se consideran bastante irrelevantes, fallando al intentar replicar la trama de la obra original.

2. The Lord of the Rings: The Fellowship of the Ring

Los resultados obtenidos con estas se considerán casi ideales, el modelo ha conseguido generar una coincidencia casi perfecta. Las recomendaciones se centran en las secuelas directas y en 'El Hobbit' siendo muy coherente en relación a la trama de la película original. Esto se debe a que existe una alta densidad de términos específicos y recurrentes como los nombres de los personajes, los lugares, etc.

3. Inception

Se pueden considerar bastantes buenas recomendaciones ya que, aunquen no logra encontrar relaciones similares con la trama, el modelo engloba películas bajo el mismo director (Christopher Nolan) y actores recurrenes (Michael Caine, Cillian Murphy) al recomendar películas como 'The Dark Knight' o 'Dunkirk'.

4. Blade Runner 2049

Aquí el modelo falla completamente. Ha sido incapaz de relacionar términos de ciencia ficción perdiendose en descripciones genéricas que le han llevado a recomendar películas como 'Cobra Kao' (artes marciales) o Outer Banks (drama juvenil)

5. Seven

Al igual que con 'Blade Runner 2049', las recomendaciones que ha dado el modelo para esta película son muy malas. Ninguna recomendación encaja con el tono de suspense criminal y falla claramente al sugerir como mejor opcion la comida romántica de ' Rainy Day in New York'. Probablemente el modelo le dió mucha importancia a la palabra 'New York' o a términos urbanos comunes en las sinopsis.


###Análsis de series

1. The Office

El modelo presenta resultados mixtos. Por un lado, acierta con 'David Brent: Life on the Road' al identificar términos vinculados al director y al universo de la obra original, y con 'Detectorists' por su género de comedia. Sin embargo, la inclusión de 'The Merchant of Venice' evidencia una desviación clara.

2. The Mandalorian

En este caso, el recomendador logra establecer conexiones coherentes dentro del ecosistema de la franquicia de Star Wars y el género espacial, como sucede con 'Marvel's Rocket & Groot'. No obstante, la aparición de The Last Czars (drama histórico) supone un error de contextualización.

3. Chernobyl

Los resultados, en este caso, son unos de las más sólidos. La recomendación de 'Meltdown: Three Mile Island' es excelente, demostrando que el TF-IDF fue muy preciso al rastrear vocabulario técnico sobre desastres nucleares. Además, títulos como 'The Liberator' mantienen el perfil de drama histórico y tensión. En este escenario, el sistema actúa muy bien porque el dataset contiene términos muy específicos y un contexto histórico delimitado que facilita la coincidencia de vectores.

4. The Boys

A excepción de 'The Boys Presents: Diabolical', que pertenece al mismo universo, el resto de las opciones son muy genéricas. El modelo se identificó solo términos de "acción" en títulos como 'Wu Assassins', pero fallando totalmente al detectar el tono de sátira política y cinismo.

5. Stranger Things

Los resultados muestran una lista muy sólida. Títulos como 'Archive 81' y 'Just Beyond' capturan con precisión la atmósfera de misterio sobrenatural y terror. Aquí el modelo funcionó de manera óptima consiguiendo agrupar palabras clave del género fantástico, logrando recomendaciones similares que coinciden con la experiencia estética y el suspenso de la serie original.

##Evaluación automática basada en géneros

En este apartado se realizará una aproximación a una evaluación "objetiva". El procedimiento es el siguiente:

- Se preparará un dataset de evaluación seleccionando aleatoriamente entre 200 y 300 títulos.
- Para cada título obtenemos un Top-K de recomendaciones (en este 10).
- Luego, comparamos los géneros del título base con los de cada recomendación.
  - Si al menos un género coincide, consideramos que la recomendación es relevante.
  - La métrica utilizada es la Precision@K, que se calcula como la proporción de recomendaciones con al menos un género común dividida por K.

Este apartado fue realizado con el apoyo de Gemini, que me proprocionó una guía para la correcta la implementación de este proceso de evaluación. Ya que se usa una métrica y una evaluación "especial" para poder simular un escenario de análisis robusto. Esto permitiría obtener así una valoración más precisa del rendimiento del modelo.


PROMPT:

Necesito implementar una evaluación automática para mi modelo recomendador TF-IDF basada en la similitud de géneros, que guía podría seguir para hacerlo?

RESPUESTA:
1. Prepare los datos: Convirtiendo la columna 'genres' de df_model en una lista de géneros (genre_list) y manejando los valores nulos o vacíos.
2. Seleccione títulos para la evaluación: Elija aleatoriamente los títulos de df_model que tengan géneros definidos.
3. Implemente el bucle de evaluación: Para cada título seleccionado, que obtenga las 10 mejores recomendaciones (Top-K donde K=10) usando la función get_recommendations que ya tenemos.
4. Calcule la relevancia: Considere una recomendación relevante si comparte al menos un género con el título original.
1. Calcule y muestre la métrica: Finalmente, que calcule la Precision@K promedio para todos los títulos evaluados y muestre un resumen claro de los resultados."

### Preparación de los datos

Se preprocesarán los datos de la columna 'genres' para convertir la cadena en una lista de cadenas y manejar los valores faltantes.


La función "parse_genres" fue implementada mediante el asistente de código de gemini.


In [7]:
import numpy as np

def parse_genres(genres_str):
    if pd.isna(genres_str) or genres_str == '':
        return []
    return [genre.strip() for genre in genres_str.split(',')]

df_model['genre_list'] = df_model['genres'].apply(parse_genres)

print("Primeras 5 filas de df_model con 'title', 'genres' y 'genre_list':")
print(df_model[['title', 'genres', 'genre_list']].head())

print("\nPrimeros 5 títulos con una 'genre_list' vacía:")
empty_genre_titles = df_model[df_model['genre_list'].apply(len) == 0]['title'].head()
print(empty_genre_titles)

Primeras 5 filas de df_model con 'title', 'genres' y 'genre_list':
                         title  \
0            The Three Stooges   
1                  The General   
2  The Best Years of Our Lives   
3              His Girl Friday   
4            In a Lonely Place   

                                              genres  \
0  comedy, family, animation, action, fantasy, ho...   
1      action, drama, war, western, comedy, european   
2                                romance, war, drama   
3                             comedy, drama, romance   
4                           thriller, drama, romance   

                                          genre_list  
0  [comedy, family, animation, action, fantasy, h...  
1    [action, drama, war, western, comedy, european]  
2                              [romance, war, drama]  
3                           [comedy, drama, romance]  
4                         [thriller, drama, romance]  

Primeros 5 títulos con una 'genre_list' vacía:
79           

Ahora que la columna genre_list está creada, se seleccionarán los títulos para la evaluación automática. Esto implica filtrar los títulos sin géneros definidos y luego elegir aleatoriamente un número específico de estos títulos para formar el conjunto de evaluación. Para esta evaluación se usarán 250 títulos diferentes.





In [8]:
df_eval_candidates = df_model[df_model['genre_list'].apply(len) > 0]

num_evaluation_titles = 250
if len(df_eval_candidates) < num_evaluation_titles:
    num_evaluation_titles = len(df_eval_candidates)

evaluation_titles_df = df_eval_candidates.sample(n=num_evaluation_titles, random_state=42)
evaluation_titles = evaluation_titles_df['title'].tolist()

print(f"Seleccionados {len(evaluation_titles)} títulos para la evaluación automática basada en géneros.")
print("Primeros 5 títulos de evaluación:")
print(evaluation_titles[:5])

Seleccionados 250 títulos para la evaluación automática basada en géneros.
Primeros 5 títulos de evaluación:
['Random Acts of Flyness', 'The Nutcrackers', 'Record of Youth', 'Jean-François and the Meaning of Life', 'Coming 2 America']


Con los títulos de evaluación y la genre_list preparados, ahora ya se puede implementar el bucle de evaluación automática para calcular Precision@K. Esto implica iterar a través de cada título de evaluación, obtener sus recomendaciones, comparar géneros para determinar la relevancia y luego calcular la Precision@K promedio.



In [9]:
K = 10 # Define K para Precision@K, lo que significa que consideramos las 10 mejores recomendaciones

precision_scores = []
evaluated_count = 0

# Crea un mapeo de título a genre_list para una búsqueda eficiente
title_to_genre_list = df_model.set_index('title')['genre_list']

print(f"\n--- Iniciando Evaluación Automática (K={K}) ---")

for title_to_evaluate in evaluation_titles:
    # Obtiene la lista de géneros para el título base
    base_title_genres = title_to_genre_list.get(title_to_evaluate, [])

    # Recupera las top-K recomendaciones para el título actual
    recommendations_df = get_recommendations(title_to_evaluate, cosine_sim_matrix, df_model, title_to_index, top_n=K)

    # Omite si el título no fue encontrado por la función de recomendación
    if isinstance(recommendations_df, str):
        # print(f"Advertencia: Saltando '{title_to_evaluate}' ya que no fue encontrado por el recomendador.")
        continue

    relevant_recommendations = 0
    # Itera a través de cada título recomendado
    for _, row in recommendations_df.iterrows():
        recommended_title = row['Recommended Title']
        # Obtiene la lista de géneros para el título recomendado usando la búsqueda eficiente
        recommended_genres = title_to_genre_list.get(recommended_title, [])

        # Verifica si hay al menos un género común entre el título base y el recomendado
        # Una recomendación se considera 'relevante' si comparte algún género
        if any(genre in base_title_genres for genre in recommended_genres if genre): # Asegura que el género no sea una cadena vacía
            relevant_recommendations += 1

    # Calcula Precision@K para el título actual
    # Precision@K = (Número de recomendaciones relevantes) / K
    precision_at_k = relevant_recommendations / K if K > 0 else 0
    precision_scores.append(precision_at_k)
    evaluated_count += 1

# Calcula la Precision@K promedio en todos los títulos evaluados
if evaluated_count > 0:
    average_precision_at_k = np.mean(precision_scores)
else:
    average_precision_at_k = 0

print(f"\n--- Resultados de la Evaluación Automática ---")
print(f"Total de títulos evaluados: {evaluated_count}")
print(f"Precision@{K} promedio: {average_precision_at_k:.4f}")


--- Iniciando Evaluación Automática (K=10) ---

--- Resultados de la Evaluación Automática ---
Total de títulos evaluados: 250
Precision@10 promedio: 0.7004


El sistema obtuvo una Precision@10 promedio de 0.7004. En términos técnicos, esto indica que el 70.04% de las recomendaciones generadas por el modelo comparten, al menos, una etiqueta de género con el título original.

Desde un punto de vista de ingeniería de datos, este resultado muestra:

- Consistencia de Género: El modelo es muy eficaz manteniendo la categoría principal (por ejemplo, si buscas una serie de "Ciencia Ficción", 7 de cada 10 resultados serán de ese género).

- Punto de Referencia: Un valor superior al 0.70 es una base sólida para un recomendador basado puramente en frecuencias.

##Resumen de resultados y conclusiones

Gracias a este proyecto se ha podido crear un modelo de recomendación para péliculas y series y el modelo desarrolado ha demostrado ser una herramienta robusta para la recuperación de información, aunque con limitaciones al procesamiento puramente estadístico del lenguaje.

###1. Aspectos Positivos

El sistema alcanzó una Precision@10 promedio de 0.7004. Esto significa que el 70% de las recomendaciones mantienen la coherencia de género con el título original, validando el uso del Coseno de Similitud como una métrica de distancia efectiva para este dataset. Además, el modelo brilla en contenidos con terminología técnica o nombres propios recurrentes. Casos como The Lord of the Rings, Chernobyl o Stranger Things demuestran que, cuando hay términos distintivos, el sistema es funciona muy bien al agrupar secuelas y universos compartidos. También, en títulos como Inception el modelo fue capaz de inferir la afinidad del usuario por directores (Nolan) o actores recurrentes, aportando una capa de inteligencia más allá de sugerir exclusivamente por la trama.

###2. Limitaciones

Al basarse en frecuencia de términos, el algoritmo no comprende el tono y el contexto. Esto explica errores groseros como recomendar una pelicula de comedia romántica tras un thriller criminal (Seven). Además, el rendimiento es muy dispar y depende de la riqueza de las sinopsis. Descripciones genéricas provocan que obras complejas como se pierdan en recomendaciones superficiales, demostrando que el modelo no entiende géneros si no hay un vocabulario muy específico que los respalde.

###3. Propuestas de Mejora

Para evolucionar este sistema hacia un recomendador mas completo se proponen las siguientes líneas:

- Embeddings Contextuales: Sustituir TF-IDF por modelos tipo Transformers, como BERT, para capturar el significado real y el tono de las descripciones, evitando las confusiones entre géneros opuestos.

- Hibridación del Sistema: Combinar el filtrado basado en contenido con filtrado colaborativo (interacciones de usuarios). Esto permitiría recomendar no solo lo que se "parece" en texto, sino lo que la comunidad considera relevante.

- Refinamiento de Pesos y Métricas: Implementar una evaluación multimétrica con métricas adicionales como Recall@K o el F1-Score y ajustar el peso de los metadatos, ponderar de distintas los datos que lo forman: géneros, actores/directores, descripciones ...con la idea de reducir el ruido estadístico.

**Conclusión final:** El proyecto cumple con su función como modelo base, demostrando que el análisis de texto es capaz de estructurar un catálogo de entretenimiento de forma lógica, sentando las bases necesarias para futuras implementaciones de inteligencia artificial más profundas.